<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-03-prompt-engineering/lesson-3.6-prompt-optimization-and-dspy/notebooks/exercises-3.6.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 3.6 — Prompt Optimization & DSPy
Netsetos GenAI Course v2.0 | Module 3


In [ ]:
# Setup
import json, re
from difflib import SequenceMatcher
# pip install dspy openai google-generativeai -q


## Ex 1: LLM-as-Judge


In [ ]:
judge_prompt = '''Rate this response on 4 dimensions (1-5):
- Relevance, Completeness, Tone, Actionability
Query: "{query}"
Response: "{response}"
Return JSON: {{"relevance":int,"completeness":int,"tone":int,"actionability":int,"total":int}}'''

tests = [
    {"query":"Where is my order?","response":"Order #4521 is at Hyderabad hub. Expected tomorrow 6PM. Added ₹100 credit."},
    {"query":"Where is my order?","response":"Check the tracking page."},
]
for t in tests:
    print(f'Testing: {t["response"][:50]}...')


## Ex 2: Eval Framework


In [ ]:
def auto_score(response, keywords):
    hits = sum(1 for k in keywords if k.lower() in response.lower())
    return hits / len(keywords)

tests = [
    {"input":"Return policy?","keywords":["30 days","refund","receipt"]},
    {"input":"Screen cracked","keywords":["warranty","repair","service center"]},
]
for t in tests:
    score = auto_score('We offer 30 day returns with receipt for refund', t['keywords'])
    print(f'{score:.0%}: {t["input"]}')


## Ex 3: A/B Test


In [ ]:
prompt_a = 'Answer the customer question.'
prompt_b = '''Answer the customer. Rules:\n1. Acknowledge concern\n2. Give specific solution\n3. Mention policy (30-day return)\n4. End with next step'''
# TODO: Run both on test set, compare scores


## Ex 4: DSPy Basics


In [ ]:
import dspy
# dspy.configure(lm=dspy.LM('openai/gpt-4o-mini'))

class Classify(dspy.Signature):
    '''Classify product review sentiment.'''
    review: str = dspy.InputField()
    sentiment: str = dspy.OutputField(desc='positive, negative, or neutral')

predict = dspy.Predict(Classify)
# result = predict(review='Amazing biryani masala!')
# print(result.sentiment)


## Ex 5: DSPy ChainOfThought


In [ ]:
cot = dspy.ChainOfThought(Classify)
# result = cot(review='Good phone but camera is poor')
# print(f'Reasoning: {result.reasoning}')
# print(f'Sentiment: {result.sentiment}')


## Ex 6: DSPy BootstrapFewShot


In [ ]:
trainset = [
    dspy.Example(review='Best phone ever!', sentiment='positive').with_inputs('review'),
    dspy.Example(review='Broke after 1 week', sentiment='negative').with_inputs('review'),
    dspy.Example(review='Decent for price', sentiment='neutral').with_inputs('review'),
    # Add 7 more...
]
def accuracy(example, prediction, trace=None):
    return prediction.sentiment.strip().lower() == example.sentiment.lower()

optimizer = dspy.BootstrapFewShot(metric=accuracy, max_bootstrapped_demos=3)
# optimized = optimizer.compile(dspy.Predict(Classify), trainset=trainset)


## Ex 7: Eval Metrics Suite


In [ ]:
def exact_match(pred, truth): return pred.strip().lower() == truth.strip().lower()
def fuzzy(pred, truth): return SequenceMatcher(None, pred.lower(), truth.lower()).ratio()
def keyword_cov(pred, kws): return sum(1 for k in kws if k.lower() in pred.lower()) / len(kws)
def format_ok(pred):
    try: json.loads(pred); return True
    except: return False

sample = '{"sentiment":"positive","rating":4}'
print(f'Exact match: {exact_match(sample, "positive")}')
print(f'Format OK: {format_ok(sample)}')
print(f'Keyword: {keyword_cov(sample, ["positive","rating"]):.0%}')


## Ex 8: Full Pipeline


In [ ]:
# The complete eval→optimize→deploy pipeline
# 1. Define eval metrics
# 2. Benchmark baseline prompt
# 3. Optimize with DSPy BootstrapFewShot
# 4. Compare optimized vs baseline on held-out test set
# 5. If improved, create promptfooconfig.yaml
# 6. Run CI/CD eval → deploy

print('Pipeline stages:')
for i, stage in enumerate(['Eval metrics defined', 'Baseline benchmarked',
    'DSPy optimized', 'Compared on held-out', 'Promptfoo config created', 'CI/CD deployed'], 1):
    print(f'  {i}. {stage}')


---
## Recap
The production workflow: measure (eval framework) → optimize (DSPy) → test (Promptfoo CI/CD) → deploy (canary) → monitor (drift).
